# MVP de Engenharia de Dados
## Análise de Acidentes em Rodovias Federais Brasileiras em 2025

**Nome:** Delianne Fernandes Purgato  
**Matrícula:** 4052026000610  
**Data:** 24/09/2026  
**Disciplina:** Engenharia de Dados  
**Plataforma:** Databricks  
**Fonte dos dados:** Polícia Rodoviária Federal (PRF)

---

## 1. Introdução

Os acidentes de trânsito representam um importante problema para a segurança viária, podendo resultar em vítimas, impactos sociais e prejuízos materiais. A análise de dados relacionados a essas ocorrências permite identificar padrões e características que podem contribuir para uma melhor compreensão do cenário dos acidentes nas rodovias.

Este MVP tem como foco a construção de uma solução de Engenharia de Dados utilizando a plataforma Databricks, a partir de dados públicos de acidentes disponibilizados pela Polícia Rodoviária Federal (PRF).

O projeto contemplará as etapas de coleta e ingestão dos dados, persistência em nuvem, análise da qualidade, transformação, modelagem e disponibilização das informações para análise.

## 2. Objetivo

O objetivo deste MVP é construir uma solução de Engenharia de Dados no Databricks para coletar, armazenar, tratar, modelar e analisar os dados de acidentes ocorridos nas rodovias federais brasileiras em 2025, disponibilizados pela Polícia Rodoviária Federal (PRF).

A partir da organização e preparação desses dados, busca-se identificar padrões relacionados à ocorrência e à gravidade dos acidentes e produzir informações que auxiliem na compreensão do cenário da segurança viária nas rodovias federais.



### 2.1 Perguntas de negócio

Para atingir o objetivo proposto, pretende-se responder às seguintes questões:

1. Quais são as principais causas dos acidentes registrados nas rodovias federais em 2025?
2. Em quais dias da semana e horários ocorre o maior número de acidentes?
3. Quais estados e rodovias federais concentram o maior número de acidentes?
4. Quais tipos de acidentes apresentam os maiores números de vítimas fatais?
5. Existe relação entre as condições meteorológicas, características da pista e a ocorrência de acidentes?
6. Quais características se destacam entre as ocorrências com vítimas fatais?

## 3. Fonte dos Dados

Para o desenvolvimento deste MVP foi utilizado o conjunto de dados de acidentes de trânsito disponibilizado pela **Polícia Rodoviária Federal (PRF)** em seu Portal de Dados Abertos.

Foi selecionada a base de **acidentes agrupados por ocorrência referente ao ano de 2025**, disponibilizada em formato CSV. Cada registro representa uma ocorrência de acidente em rodovia federal e contém informações relacionadas à data e horário, localização, características da via, condições meteorológicas, causa e tipo do acidente, além da quantidade de pessoas, veículos, feridos e mortos envolvidos.

**Fonte:** Polícia Rodoviária Federal (PRF) – Dados Abertos  
**Período analisado:** 2025  
**Formato de origem:** CSV  
**Arquivo:** datatran2025.csv  
**Link da fonte:** [Portal de Dados Abertos da PRF](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf)  
**Condições de uso:** Dados abertos de uso livre, permitindo utilização, reutilização e redistribuição, conforme a Política de Dados Abertos da PRF.

Os dados disponibilizados pela PRF são dados públicos governamentais. A origem e a documentação das variáveis serão consideradas também na elaboração do catálogo e da linhagem dos dados deste projeto.

## 4. Coleta e Ingestão dos Dados

Os dados utilizados neste projeto foram obtidos a partir do conjunto de Dados Abertos da Polícia Rodoviária Federal (PRF), considerando os acidentes agrupados por ocorrência referentes ao ano de 2025.

O arquivo de origem, disponibilizado em formato CSV, foi ingerido no ambiente Databricks e persistido na plataforma de nuvem para utilização nas etapas posteriores do pipeline de dados.

Durante a ingestão, foi identificado que o arquivo utiliza ponto e vírgula (`;`) como delimitador. Essa configuração foi informada ao Databricks para que os atributos fossem corretamente reconhecidos e separados em colunas. E também foi necessário ajustar a codificação do arquivo para Windows-1252, garantindo a correta apresentação dos caracteres acentuados.

Os dados originais foram armazenados na tabela:

'workspace.bronze.bronze_acidentes_prf_2025'

A tabela Bronze representa a camada de dados brutos do projeto, preservando os dados provenientes da fonte antes da aplicação das regras de tratamento, qualidade e modelagem.

###4.1 Preparação do ambiente

Para organização das camadas da arquitetura medalhão, foram criados no catálogo 'workspace' três schemas distintos: 'bronze', 'silver' e 'gold'. Essa separação permite armazenar os dados de acordo com o estágio de processamento dentro do pipeline.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
bronze
default
gold
information_schema
silver


### 4.2 Validação da Ingestão

Após a criação da tabela Bronze, foi realizada uma validação da carga para verificar se os dados foram persistidos corretamente no Databricks.

Inicialmente, foi verificada a quantidade total de registros armazenados. A tabela `workspace.bronze.bronze_acidentes_prf_2025` apresentou **72.529 registros**.

Em seguida, foi realizada uma consulta de amostragem dos dados para verificar a estrutura das informações carregadas e confirmar a disponibilidade dos atributos para as próximas etapas do pipeline.


In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM workspace.bronze.bronze_acidentes_prf_2025;

total_registros
72529


In [0]:
%sql
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN id COMMENT 'Identificador único da ocorrência de acidente';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN data_inversa COMMENT 'Data da ocorrência do acidente';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN dia_semana COMMENT 'Dia da semana em que ocorreu o acidente';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN horario COMMENT 'Horário da ocorrência do acidente';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN uf COMMENT 'Unidade federativa onde ocorreu o acidente';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN br COMMENT 'Número da rodovia federal';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN km COMMENT 'Quilômetro da rodovia onde ocorreu o acidente';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN municipio COMMENT 'Município onde ocorreu o acidente';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN causa_acidente COMMENT 'Causa identificada para a ocorrência do acidente';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN tipo_acidente COMMENT 'Tipo de acidente registrado';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN classificacao_acidente COMMENT 'Classificação do acidente quanto à gravidade';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN fase_dia COMMENT 'Fase do dia no momento da ocorrência';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN sentido_via COMMENT 'Sentido da via no local do acidente';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN condicao_metereologica COMMENT 'Condição meteorológica registrada no momento do acidente';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN tipo_pista COMMENT 'Tipo de pista da rodovia';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN tracado_via COMMENT 'Característica do traçado da via';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN uso_solo COMMENT 'Indicação de ocorrência em área urbana ou rural';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN pessoas COMMENT 'Quantidade total de pessoas envolvidas na ocorrência';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN mortos COMMENT 'Quantidade de pessoas mortas na ocorrência';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN feridos_leves COMMENT 'Quantidade de pessoas com ferimentos leves';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN feridos_graves COMMENT 'Quantidade de pessoas com ferimentos graves';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN ilesos COMMENT 'Quantidade de pessoas ilesas';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN ignorados COMMENT 'Quantidade de pessoas com estado físico não informado';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN feridos COMMENT 'Quantidade total de pessoas feridas';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN veiculos COMMENT 'Quantidade de veículos envolvidos na ocorrência';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN latitude COMMENT 'Latitude geográfica do local da ocorrência';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN longitude COMMENT 'Longitude geográfica do local da ocorrência';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN regional COMMENT 'Regional da Polícia Rodoviária Federal responsável pela ocorrência';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN delegacia COMMENT 'Delegacia da Polícia Rodoviária Federal responsável pela ocorrência';
ALTER TABLE workspace.bronze.bronze_acidentes_prf_2025 ALTER COLUMN uop COMMENT 'Unidade Operacional da Polícia Rodoviária Federal responsável pela ocorrência';

#### 4.2.1 Estrutura dos dados de origem

A estrutura da tabela Bronze foi validada após a ingestão para identificar os atributos e os tipos de dados provenientes do arquivo de origem 'datatran2025.csv'.


In [0]:
%sql
DESCRIBE TABLE workspace.bronze.bronze_acidentes_prf_2025;

col_name,data_type,comment
id,string,Identificador único da ocorrência de acidente
data_inversa,string,Data da ocorrência do acidente
dia_semana,string,Dia da semana em que ocorreu o acidente
horario,string,Horário da ocorrência do acidente
uf,string,Unidade federativa onde ocorreu o acidente
br,string,Número da rodovia federal
km,string,Quilômetro da rodovia onde ocorreu o acidente
municipio,string,Município onde ocorreu o acidente
causa_acidente,string,Causa identificada para a ocorrência do acidente
tipo_acidente,string,Tipo de acidente registrado


In [0]:
%sql
SELECT *
FROM workspace.bronze.bronze_acidentes_prf_2025
LIMIT 10;


id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
652493,2025-01-01,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP
652519,2025-01-01,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,NA,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE
652522,2025-01-01,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR
652544,2025-01-01,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR
652549,2025-01-01,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG
652569,2025-01-01,quarta-feira,10:40:00,MT,70,669,CACERES,Transitar na contramão,Colisão frontal,Com Vítimas Fatais,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,4,2,0,0,1,2,0,5,"-16,04148578","-57,25884017",SPRF-MT,DEL03-MT,UOP02-DEL03-MT
652573,2025-01-01,quarta-feira,12:23:00,RS,116,376,TAPES,Ausência de reação do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Dupla,Reta,Não,2,0,1,0,0,1,1,2,"-30,739714","-51,62594",SPRF-RS,DEL02-RS,UOP02-DEL02-RS
652617,2025-01-01,quarta-feira,17:45:00,SC,101,"207,4",SAO JOSE,Ausência de reação do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Nublado,Dupla,Reta,Sim,2,0,1,0,1,0,1,2,"-27,60001226","-48,6226467",SPRF-SC,DEL01-SC,UOP01-DEL01-SC
652625,2025-01-01,quarta-feira,18:40:00,MG,116,"708,5",MURIAE,Velocidade Incompatível,Tombamento,Com Vítimas Fatais,Anoitecer,Crescente,Nublado,Simples,Curva,Não,2,1,0,0,0,1,0,2,"-21,16328873","-42,37968988",SPRF-MG,DEL07-MG,UOP02-DEL07-MG
652648,2025-01-01,quarta-feira,17:00:00,PE,407,"7,4",AFRANIO,Demais falhas mecânicas ou elétricas,Incêndio,Sem Vítimas,Pleno dia,Crescente,Céu Claro,Simples,Aclive;Curva,Não,2,0,0,0,2,0,0,1,"-8,47503105","-41,0137105",SPRF-PE,DEL06-PE,UOP02-DEL06-PE


In [0]:
%sql
SELECT DISTINCT dia_semana
FROM workspace.bronze.bronze_acidentes_prf_2025
ORDER BY dia_semana;

dia_semana
domingo
quarta-feira
quinta-feira
segunda-feira
sexta-feira
sábado
terça-feira


In [0]:
df_bronze = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "windows-1252")
    .option("inferSchema", "false")
    .csv("/Volumes/workspace/default/arquivo_mvp/datatran2025.csv")
)

df_bronze.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.bronze_acidentes_prf_2025")

In [0]:
%sql
SELECT DISTINCT dia_semana
FROM workspace.bronze.bronze_acidentes_prf_2025
ORDER BY dia_semana;

dia_semana
domingo
quarta-feira
quinta-feira
segunda-feira
sexta-feira
sábado
terça-feira


## 5. Modelagem


### 5.1 Arquitetura de Dados
O projeto foi estruturado no Databricks utilizando uma arquitetura em camadas:

- **Bronze:** armazenamento dos dados provenientes do arquivo de origem;
- **Silver:** dados tratados e padronizados para utilização nas etapas seguintes;
- **Gold:** dados organizados em modelo dimensional para realização das análises.

Essa organização permite separar os dados de origem, os processos de transformação e a camada destinada ao consumo analítico.

###5.2 Modelo Dimensional
A Camada Gold foi estruturada utilizando um modelo dimensional em esquema estrela, composto pela tabela fato_acidentes e pelas dimensões de tempo, localidade, via e acidente.

A figura abaixo apresenta a estrutura do modelo dimensional implementado.

![Modelo Dimensional](/Volumes/workspace/default/arquivo_mvp/modelo_dimensional.png)

###5.3 Catálogo de Dados
O catálogo de dados apresenta os principais atributos utilizados no modelo dimensional, seus significados, tipos e domínios. Para os atributos numéricos são considerados os valores mínimos e máximos encontrados no conjunto de dados, enquanto para os atributos categóricos são apresentadas as categorias existentes.

#### 5.3.1 Contexto das tabelas

O modelo dimensional da Camada Gold é composto por uma tabela fato e quatro dimensões:

- **fato_acidentes:** concentra as métricas relacionadas às ocorrências de acidentes e as chaves de relacionamento com as dimensões.
- **dim_tempo:** organiza os atributos relacionados à data das ocorrências.
- **dim_localidade:** contém as informações geográficas de unidade federativa e município.
- **dim_via:** reúne as características da rodovia e da via onde ocorreu o acidente.
- **dim_acidente:** concentra as características relacionadas à causa, tipo, classificação e condições do acidente.

#### Catálogo do Modelo Dimensional

| Tabela | Atributo | Descrição | Tipo | Domínio / Valores |
|---|---|---|---|---|
| dim_tempo | tempo_key | Chave da dimensão tempo | Numérico | Chave técnica |
| dim_tempo | data | Data da ocorrência | Data | Ano de 2025 |
| dim_tempo | dia_semana | Dia da semana da ocorrência | Categórico | segunda-feira a domingo |
| dim_tempo | ano | Ano da ocorrência | Numérico | 2025 |
| dim_tempo | mes | Mês da ocorrência | Numérico | Mín. 1 / Máx. 12 |
| dim_tempo | dia | Dia do mês da ocorrência | Numérico | Mín. 1 / Máx. 31 |
| dim_localidade | localidade_key | Chave da dimensão localidade | Numérico | Chave técnica |
| dim_localidade | uf | Unidade Federativa | Categórico | AC, AL, AM, AP, BA, CE, DF, ES, GO, MA, MG, MS, MT, PA, PB, PE, PI, PR, RJ, RN, RO, RR, RS, SC, SE, SP, TO |
| dim_localidade | municipio | Município da ocorrência | Categórico | Municípios presentes na base |
| dim_via | via_key | Chave da dimensão via | Numérico | Chave técnica |
| dim_via | br | Número da rodovia federal | Numérico | Mín. 0 / Máx. 495 |
| dim_via | km | Quilômetro da ocorrência | Numérico | Mín. 0,00 / Máx. 1.257,00 |
| dim_via | sentido_via | Sentido da via | Categórico | Crescente, Decrescente, Não Informado |
| dim_via | tipo_pista | Tipo da pista | Categórico | Dupla, Múltipla, Simples |
| dim_via | tracado_via | Características do traçado da via | Categórico multivalorado | 605 combinações observadas, incluindo Reta, Curva, Declive, Aclive, Interseção de Vias e combinações entre essas características |
| dim_via | uso_solo | Indicação de uso do solo | Categórico | Não, Sim |
| dim_acidente | acidente_key | Chave da dimensão acidente | Numérico | Chave técnica |
| dim_acidente | causa_acidente | Causa registrada para o acidente | Categórico | Causas presentes na base |
| dim_acidente | tipo_acidente | Tipo do acidente | Categórico | Tipos presentes na base |
| dim_acidente | classificacao_acidente | Classificação do acidente | Categórico | Com Vítimas Fatais, Com Vítimas Feridas, NA, Sem Vítimas |
| dim_acidente | fase_dia | Fase do dia | Categórico | Amanhecer, Anoitecer, Plena Noite, Pleno dia |
| dim_acidente | condicao_metereologica | Condição meteorológica | Categórico | Chuva, Céu Claro, Garoa/Chuvisco, Ignorado, Neve, Nevoeiro/Neblina, Nublado, Sol, Vento |
| fato_acidentes | acidente_id | Identificador da ocorrência | Numérico | Identificador |
| fato_acidentes | tempo_key | Chave de relacionamento com dim_tempo | Numérico | Chave estrangeira |
| fato_acidentes | localidade_key | Chave de relacionamento com dim_localidade | Numérico | Chave estrangeira |
| fato_acidentes | via_key | Chave de relacionamento com dim_via | Numérico | Chave estrangeira |
| fato_acidentes | acidente_key | Chave de relacionamento com dim_acidente | Numérico | Chave estrangeira |
| fato_acidentes | horario | Horário da ocorrência | Categórico | HH:mm:ss |
| fato_acidentes | pessoas | Quantidade de pessoas envolvidas | Numérico | Mín. 1 / Máx. 76 |
| fato_acidentes | mortos | Quantidade de mortos | Numérico | Mín. 0 / Máx. 16 |
| fato_acidentes | feridos_leves | Quantidade de feridos leves | Numérico | Mín. 0 / Máx. 41 |
| fato_acidentes | feridos_graves | Quantidade de feridos graves | Numérico | Mín. 0 / Máx. 22 |
| fato_acidentes | ilesos | Quantidade de pessoas ilesas | Numérico | Mín. 0 / Máx. 71 |
| fato_acidentes | ignorados | Quantidade de pessoas com condição ignorada | Numérico | Mín. 0 / Máx. 81 |
| fato_acidentes | feridos | Total de feridos | Numérico | Mín. 0 / Máx. 49 |
| fato_acidentes | veiculos | Quantidade de veículos envolvidos | Numérico | Mín. 1 / Máx. 82 |

#### 5.3.2 Evidência do catálogo no Databricks

As tabelas utilizadas no pipeline foram persistidas no catálogo workspace do Databricks, organizadas em schemas distintos de acordo com as camadas da arquitetura medalhão. A Camada Bronze foi armazenada no schema bronze, a Camada Silver no schema silver e as dimensões e a tabela fato da Camada Gold no schema gold.


![Tabelas persistidas](/Volumes/workspace/default/arquivo_mvp/Tabela_persistida_new.png)

###5.4 Linhagem dos Dados
Os dados utilizados no projeto foram obtidos no Portal de Dados Abertos da Polícia Rodoviária Federal (PRF), por meio do arquivo 'datatran2025.csv'.

O arquivo foi armazenado no Databricks e utilizado na criação da Camada Bronze. Em seguida, os dados foram tratados e padronizados na Camada Silver e posteriormente organizados na Camada Gold, por meio das dimensões e da tabela fato utilizadas nas análises.

O fluxo de dados do projeto pode ser representado da seguinte forma:

![Linhagem dos Dados](/Volumes/workspace/default/arquivo_mvp/linhagem_dados.png)

## 6. Carga dos Dados
Nesta etapa, são documentados os processos de carga e transformação dos dados ao longo das camadas Bronze, Silver e Gold. A partir dos dados de origem, o pipeline realiza a persistência dos dados brutos, o tratamento e a padronização das informações e, posteriormente, a construção do modelo dimensional utilizado nas análises.

O pipeline de dados foi implementado em um único notebook no Databricks, utilizando SQL e organizado de forma sequencial. O fluxo contempla a ingestão e persistência dos dados na Camada Bronze, o tratamento e a padronização na Camada Silver e a construção do modelo dimensional na Camada Gold. As etapas e os respectivos códigos de transformação e carga estão documentados ao longo desta seção.



###6.1 Carga na Camada Bronze
A carga inicial dos dados foi realizada a partir do arquivo 'datatran2025.csv', previamente obtido no portal de Dados Abertos da Polícia Rodoviária Federal (PRF).

O arquivo foi carregado no Databricks por meio da funcionalidade de upload de arquivos da plataforma, sendo persistido na tabela:

'workspace.bronze.bronze_acidentes_prf_2025'

A Camada Bronze foi utilizada para preservar os dados provenientes da fonte de origem antes da aplicação das transformações realizadas nas etapas seguintes do pipeline.

Após a carga, foram armazenados **72.529 registros** na tabela Bronze.

#### Evidência da persistência da tabela na Camada Bronze

A imagem abaixo apresenta a tabela 'bronze_acidentes_prf_2025' persistida no catálogo do Databricks após o processo de ingestão dos dados.

![Tabela persistida na Camada Bronze](/Volumes/workspace/default/arquivo_mvp/Bronze_persistida_new.png)


###6.2 Transformação e Carga na Camada Silver
A Camada Silver foi construída a partir dos dados armazenados na Camada Bronze, com o objetivo de padronizar os atributos e adequar os tipos de dados para as etapas posteriores do projeto.
Durante a transformação, foram realizados os seguintes tratamentos:
- conversão do campo id para tipo numérico;
- conversão do campo data_inversa para o tipo DATE;
- conversão do campo br para tipo numérico;
- padronização do campo horario no formato HH:mm:ss;
- conversão do campo km para o tipo decimal, substituindo a vírgula pelo ponto;
- conversão dos campos quantitativos pessoas, mortos, feridos_leves, feridos_graves, ilesos, ignorados, feridos e veiculos para tipo inteiro;
- conversão dos campos latitude e longitude para o tipo decimal;
- inclusão do atributo silver_load_timestamp para registrar o momento de processamento da carga.
Após as transformações, os dados foram persistidos na tabela workspace.silver.silver_acidentes_prf_2025.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.silver_acidentes_prf_2025 AS
SELECT
    CAST(id AS BIGINT) AS id,
    CAST(data_inversa AS DATE) AS data_inversa,
    dia_semana,
    horario,
    uf,
    CAST(br AS INT) AS br,
    CAST(REPLACE(km, ',', '.') AS DECIMAL(10,2)) AS km,
    municipio,
    causa_acidente,
    tipo_acidente,
    classificacao_acidente,
    fase_dia,
    sentido_via,
    condicao_metereologica,
    tipo_pista,
    tracado_via,
    uso_solo,
    CAST(pessoas AS INT) AS pessoas,
    CAST(mortos AS INT) AS mortos,
    CAST(feridos_leves AS INT) AS feridos_leves,
    CAST(feridos_graves AS INT) AS feridos_graves,
    CAST(ilesos AS INT) AS ilesos,
    CAST(ignorados AS INT) AS ignorados,
    CAST(feridos AS INT) AS feridos,
    CAST(veiculos AS INT) AS veiculos,
    CAST(REPLACE(latitude, ',', '.') AS DECIMAL(10,6)) AS latitude,
    CAST(REPLACE(longitude, ',', '.') AS DECIMAL(10,6)) AS longitude,
    regional,
    delegacia,
    uop,
    current_timestamp() AS silver_load_timestamp
FROM workspace.bronze.bronze_acidentes_prf_2025;

num_affected_rows,num_inserted_rows


In [0]:
%sql
DESCRIBE TABLE workspace.silver.silver_acidentes_prf_2025;

col_name,data_type,comment
id,bigint,null
data_inversa,date,null
dia_semana,string,null
horario,string,null
uf,string,null
br,int,null
km,"decimal(10,2)",null
municipio,string,null
causa_acidente,string,null
tipo_acidente,string,null


###6.3 Carga na Camada Gold
A Camada Gold foi estruturada utilizando o modelo dimensional em esquema estrela, com a criação de dimensões e uma tabela fato a partir dos dados tratados na Camada Silver.

O modelo tem como objetivo organizar os dados para facilitar as análises relacionadas aos acidentes registrados nas rodovias federais em 2025.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_tempo AS
SELECT DISTINCT
    CAST(date_format(data_inversa, 'yyyyMMdd') AS INT) AS tempo_key,
    data_inversa AS data,
    dia_semana,
    YEAR(data_inversa) AS ano,
    MONTH(data_inversa) AS mes,
    DAY(data_inversa) AS dia
FROM workspace.silver.silver_acidentes_prf_2025;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.gold.dim_tempo
ORDER BY data
LIMIT 10;

tempo_key,data,dia_semana,ano,mes,dia
20250101,2025-01-01,quarta-feira,2025,1,1
20250102,2025-01-02,quinta-feira,2025,1,2
20250103,2025-01-03,sexta-feira,2025,1,3
20250104,2025-01-04,sábado,2025,1,4
20250105,2025-01-05,domingo,2025,1,5
20250106,2025-01-06,segunda-feira,2025,1,6
20250107,2025-01-07,terça-feira,2025,1,7
20250108,2025-01-08,quarta-feira,2025,1,8
20250109,2025-01-09,quinta-feira,2025,1,9
20250110,2025-01-10,sexta-feira,2025,1,10


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_localidade AS
SELECT
    ROW_NUMBER() OVER (ORDER BY uf, municipio) AS localidade_key,
    uf,
    municipio
FROM (
    SELECT DISTINCT
        uf,
        municipio
    FROM workspace.silver.silver_acidentes_prf_2025
);


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.gold.dim_localidade
ORDER BY localidade_key
LIMIT 10;


localidade_key,uf,municipio
1,AC,ACRELANDIA
2,AC,ASSIS BRASIL
3,AC,BRASILEIA
4,AC,BUJARI
5,AC,CAPIXABA
6,AC,CRUZEIRO DO SUL
7,AC,EPITACIOLANDIA
8,AC,FEIJO
9,AC,MANOEL URBANO
10,AC,PLACIDO DE CASTRO


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_via AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY br, km, sentido_via, tipo_pista, tracado_via, uso_solo
    ) AS via_key,
    br,
    km,
    sentido_via,
    tipo_pista,
    tracado_via,
    uso_solo
FROM (
    SELECT DISTINCT
        br,
        km,
        sentido_via,
        tipo_pista,
        tracado_via,
        uso_solo
    FROM workspace.silver.silver_acidentes_prf_2025
);


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.gold.dim_via
ORDER BY via_key
LIMIT 10;

via_key,br,km,sentido_via,tipo_pista,tracado_via,uso_solo
1,0,0.00,Não Informado,Dupla,Aclive,Sim
2,0,0.00,Não Informado,Dupla,Curva,Não
3,0,0.00,Não Informado,Dupla,Curva,Sim
4,0,0.00,Não Informado,Dupla,Curva;Declive,Não
5,0,0.00,Não Informado,Dupla,Curva;Declive;Rotatória,Não
6,0,0.00,Não Informado,Dupla,Curva;Interseção de Vias,Não
7,0,0.00,Não Informado,Dupla,Curva;Interseção de Vias;Aclive,Sim
8,0,0.00,Não Informado,Dupla,Interseção de Vias,Não
9,0,0.00,Não Informado,Dupla,Interseção de Vias,Sim
10,0,0.00,Não Informado,Dupla,Interseção de Vias;Curva;Aclive,Sim


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_acidente AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY causa_acidente, tipo_acidente, classificacao_acidente,
                 fase_dia, condicao_metereologica
    ) AS acidente_key,
    causa_acidente,
    tipo_acidente,
    classificacao_acidente,
    fase_dia,
    condicao_metereologica
FROM (
    SELECT DISTINCT
        causa_acidente,
        tipo_acidente,
        classificacao_acidente,
        fase_dia,
        condicao_metereologica
    FROM workspace.silver.silver_acidentes_prf_2025
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.gold.dim_acidente
ORDER BY acidente_key
LIMIT 10;

acidente_key,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,condicao_metereologica
1,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Animal,Com Vítimas Fatais,Pleno dia,Céu Claro
2,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Animal,Com Vítimas Feridas,Pleno dia,Céu Claro
3,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Animal,Com Vítimas Feridas,Pleno dia,Nublado
4,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Animal,Sem Vítimas,Pleno dia,Céu Claro
5,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Pedestre,Com Vítimas Fatais,Amanhecer,Céu Claro
6,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Pedestre,Com Vítimas Fatais,Amanhecer,Nublado
7,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Pedestre,Com Vítimas Fatais,Anoitecer,Céu Claro
8,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Pedestre,Com Vítimas Fatais,Plena Noite,Chuva
9,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Pedestre,Com Vítimas Fatais,Plena Noite,Céu Claro
10,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Pedestre,Com Vítimas Fatais,Plena Noite,Garoa/Chuvisco


###6.3.1 Tabela Fato
A tabela fato concentra cada ocorrência de acidente e suas principais métricas, relacionando os registros às dimensões de tempo, localidade, via e acidente.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.fato_acidentes AS
SELECT
    s.id AS acidente_id,
    t.tempo_key,
    l.localidade_key,
    v.via_key,
    a.acidente_key,
    s.horario,
    s.pessoas,
    s.mortos,
    s.feridos_leves,
    s.feridos_graves,
    s.ilesos,
    s.ignorados,
    s.feridos,
    s.veiculos
FROM workspace.silver.silver_acidentes_prf_2025 s

INNER JOIN workspace.gold.dim_tempo t
    ON s.data_inversa = t.data

INNER JOIN workspace.gold.dim_localidade l
    ON s.uf = l.uf
   AND s.municipio = l.municipio

INNER JOIN workspace.gold.dim_via v
    ON s.br = v.br
   AND s.km = v.km
   AND s.sentido_via = v.sentido_via
   AND s.tipo_pista = v.tipo_pista
   AND s.tracado_via = v.tracado_via
   AND s.uso_solo = v.uso_solo

INNER JOIN workspace.gold.dim_acidente a
    ON s.causa_acidente = a.causa_acidente
   AND s.tipo_acidente = a.tipo_acidente
   AND s.classificacao_acidente = a.classificacao_acidente
   AND s.fase_dia = a.fase_dia
   AND s.condicao_metereologica = a.condicao_metereologica;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM workspace.gold.fato_acidentes;

total_registros
72529


In [0]:
%sql
SELECT *
FROM workspace.gold.fato_acidentes
LIMIT 10;

acidente_id,tempo_key,localidade_key,via_key,acidente_key,horario,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos
652493,20250101,1792,21246,6075,06:20:00,2,0,1,0,0,1,1,2
652519,20250101,297,24314,5338,07:50:00,6,1,1,0,1,4,1,6
652522,20250101,1179,49535,5867,08:45:00,5,0,3,0,2,0,3,2
652544,20250101,1160,18604,6018,11:00:00,5,0,1,0,4,0,1,2
652549,20250101,594,35827,7237,09:30:00,5,0,1,1,1,2,2,4
652569,20250101,797,6384,6516,10:40:00,4,2,0,0,1,2,0,5
652573,20250101,1607,22993,1490,12:23:00,2,0,1,0,0,1,1,2
652617,20250101,1731,12114,1337,17:45:00,2,0,1,0,1,0,1,2
652625,20250101,650,24901,7470,18:40:00,2,1,0,0,0,1,0,2
652648,20250101,982,56695,3206,17:00:00,2,0,0,0,2,0,0,1


## 7. Análise

### 7.1 Qualidade dos Dados

Após a construção da Camada Gold, foi realizada uma validação simples para verificar a consistência dos dados ao longo do pipeline e confirmar que a modelagem não resultou em perda de registros.

In [0]:
%sql
SELECT
    (SELECT COUNT(*) 
     FROM workspace.bronze.bronze_acidentes_prf_2025) AS bronze,
     
    (SELECT COUNT(*) 
     FROM workspace.silver.silver_acidentes_prf_2025) AS silver,
     
    (SELECT COUNT(*) 
     FROM workspace.gold.fato_acidentes) AS gold;

bronze,silver,gold
72529,72529,72529


A comparação entre as camadas demonstrou que os **72.529 registros** foram mantidos ao longo do pipeline, desde a Camada Bronze até a tabela fato da Camada Gold, não sendo identificada perda de registros durante as etapas de transformação e modelagem.

#### 7.1.1 Qualidade dos atributos

Para avaliar a qualidade do conjunto de dados, foram analisados os atributos da Camada Silver, verificando a presença de valores nulos e a quantidade de valores distintos em cada campo.

In [0]:
%sql

SELECT 'id' atributo,
       COUNT(*) - COUNT(id) nulos,
       COUNT(DISTINCT id) distintos
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'data_inversa', COUNT(*) - COUNT(data_inversa), COUNT(DISTINCT data_inversa)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'dia_semana', COUNT(*) - COUNT(dia_semana), COUNT(DISTINCT dia_semana)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'horario', COUNT(*) - COUNT(horario), COUNT(DISTINCT horario)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'uf', COUNT(*) - COUNT(uf), COUNT(DISTINCT uf)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'br', COUNT(*) - COUNT(br), COUNT(DISTINCT br)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'km', COUNT(*) - COUNT(km), COUNT(DISTINCT km)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'municipio', COUNT(*) - COUNT(municipio), COUNT(DISTINCT municipio)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'causa_acidente', COUNT(*) - COUNT(causa_acidente), COUNT(DISTINCT causa_acidente)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'tipo_acidente', COUNT(*) - COUNT(tipo_acidente), COUNT(DISTINCT tipo_acidente)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'classificacao_acidente', COUNT(*) - COUNT(classificacao_acidente), COUNT(DISTINCT classificacao_acidente)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'fase_dia', COUNT(*) - COUNT(fase_dia), COUNT(DISTINCT fase_dia)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'sentido_via', COUNT(*) - COUNT(sentido_via), COUNT(DISTINCT sentido_via)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'condicao_metereologica', COUNT(*) - COUNT(condicao_metereologica), COUNT(DISTINCT condicao_metereologica)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'tipo_pista', COUNT(*) - COUNT(tipo_pista), COUNT(DISTINCT tipo_pista)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'tracado_via', COUNT(*) - COUNT(tracado_via), COUNT(DISTINCT tracado_via)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'uso_solo', COUNT(*) - COUNT(uso_solo), COUNT(DISTINCT uso_solo)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'pessoas', COUNT(*) - COUNT(pessoas), COUNT(DISTINCT pessoas)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'mortos', COUNT(*) - COUNT(mortos), COUNT(DISTINCT mortos)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'feridos_leves', COUNT(*) - COUNT(feridos_leves), COUNT(DISTINCT feridos_leves)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'feridos_graves', COUNT(*) - COUNT(feridos_graves), COUNT(DISTINCT feridos_graves)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'ilesos', COUNT(*) - COUNT(ilesos), COUNT(DISTINCT ilesos)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'ignorados', COUNT(*) - COUNT(ignorados), COUNT(DISTINCT ignorados)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'feridos', COUNT(*) - COUNT(feridos), COUNT(DISTINCT feridos)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'veiculos', COUNT(*) - COUNT(veiculos), COUNT(DISTINCT veiculos)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'latitude', COUNT(*) - COUNT(latitude), COUNT(DISTINCT latitude)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'longitude', COUNT(*) - COUNT(longitude), COUNT(DISTINCT longitude)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'regional', COUNT(*) - COUNT(regional), COUNT(DISTINCT regional)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'delegacia', COUNT(*) - COUNT(delegacia), COUNT(DISTINCT delegacia)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'uop', COUNT(*) - COUNT(uop), COUNT(DISTINCT uop)
FROM workspace.silver.silver_acidentes_prf_2025

UNION ALL
SELECT 'silver_load_timestamp',
       COUNT(*) - COUNT(silver_load_timestamp),
       COUNT(DISTINCT silver_load_timestamp)
FROM workspace.silver.silver_acidentes_prf_2025;

atributo,nulos,distintos
id,0,72529
data_inversa,0,365
dia_semana,0,7
horario,0,1412
uf,0,27
br,0,115
km,0,7655
municipio,0,1844
causa_acidente,0,69
tipo_acidente,0,17


**Resultado da análise de qualidade:** Na avaliação de completude, não foram identificados valores nulos nos 31 atributos analisados da Camada Silver. A quantidade de valores distintos também se mostrou compatível com os domínios observados no conjunto de dados, como 365 datas, 7 dias da semana, 27 UFs, 4 classificações de acidente e 3 tipos de pista.

Apesar da ausência de valores nulos, foram identificados valores categóricos que representam ausência ou indisponibilidade da informação, como NA na classificação do acidente, Não Informado no sentido da via e Ignorado na condição meteorológica. Esses valores foram mantidos por representarem categorias existentes na fonte de dados.

Durante o processo de preparação dos dados, também foram realizados ajustes de tipagem e de codificação de caracteres do arquivo de origem. Esses tratamentos permitiram manter a consistência dos dados utilizados na construção do modelo dimensional e nas análises propostas.

In [0]:
%sql
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id) AS ids_distintos,
    COUNT(*) - COUNT(DISTINCT id) AS ids_duplicados
FROM workspace.silver.silver_acidentes_prf_2025;

total_registros,ids_distintos,ids_duplicados
72529,72529,0


**Unicidade:** não foram identificados registros duplicados com base no atributo id, utilizado como identificador da ocorrência. A quantidade total de registros é igual à quantidade de identificadores distintos, evidenciando a unicidade das ocorrências analisadas.

In [0]:
%sql
SELECT 'UF inválida' AS validacao, COUNT(*) AS registros
FROM workspace.silver.silver_acidentes_prf_2025
WHERE uf NOT IN (
    'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS',
    'MG','PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC',
    'SP','SE','TO'
)

UNION ALL

SELECT 'Dia da semana inválido', COUNT(*)
FROM workspace.silver.silver_acidentes_prf_2025
WHERE dia_semana NOT IN (
    'segunda-feira','terça-feira','quarta-feira',
    'quinta-feira','sexta-feira','sábado','domingo'
)

UNION ALL

SELECT 'Tipo de pista inválido', COUNT(*)
FROM workspace.silver.silver_acidentes_prf_2025
WHERE tipo_pista NOT IN ('Simples','Dupla','Múltipla');

validacao,registros
UF inválida,0
Dia da semana inválido,0
Tipo de pista inválido,0


**Consistência:** não foram identificados valores fora dos domínios esperados para os atributos uf, dia_semana e tipo_pista. As três validações retornaram zero registros inválidos, indicando consistência dos valores categóricos avaliados.

In [0]:
%sql
SELECT
    SUM(CASE WHEN pessoas < 0 THEN 1 ELSE 0 END) AS pessoas_negativas,
    SUM(CASE WHEN mortos < 0 THEN 1 ELSE 0 END) AS mortos_negativos,
    SUM(CASE WHEN feridos_leves < 0 THEN 1 ELSE 0 END) AS feridos_leves_negativos,
    SUM(CASE WHEN feridos_graves < 0 THEN 1 ELSE 0 END) AS feridos_graves_negativos,
    SUM(CASE WHEN ilesos < 0 THEN 1 ELSE 0 END) AS ilesos_negativos,
    SUM(CASE WHEN ignorados < 0 THEN 1 ELSE 0 END) AS ignorados_negativos,
    SUM(CASE WHEN feridos < 0 THEN 1 ELSE 0 END) AS feridos_negativos,
    SUM(CASE WHEN veiculos <= 0 THEN 1 ELSE 0 END) AS veiculos_invalidos
FROM workspace.silver.silver_acidentes_prf_2025;

pessoas_negativas,mortos_negativos,feridos_leves_negativos,feridos_graves_negativos,ilesos_negativos,ignorados_negativos,feridos_negativos,veiculos_invalidos
0,0,0,0,0,0,0,0


**Plausibilidade dos dados:** foram realizadas validações nos atributos quantitativos relacionados às pessoas e aos veículos envolvidos nas ocorrências. Não foram identificados valores negativos nos campos de pessoas, mortos, feridos, ilesos ou ignorados, nem valores inválidos para a quantidade de veículos, indicando valores plausíveis para esses atributos.

In [0]:
%sql
SELECT
    MIN(pessoas) AS min_pessoas,
    MAX(pessoas) AS max_pessoas,
    MIN(mortos) AS min_mortos,
    MAX(mortos) AS max_mortos,
    MIN(feridos_leves) AS min_feridos_leves,
    MAX(feridos_leves) AS max_feridos_leves,
    MIN(feridos_graves) AS min_feridos_graves,
    MAX(feridos_graves) AS max_feridos_graves,
    MIN(ilesos) AS min_ilesos,
    MAX(ilesos) AS max_ilesos,
    MIN(ignorados) AS min_ignorados,
    MAX(ignorados) AS max_ignorados,
    MIN(feridos) AS min_feridos,
    MAX(feridos) AS max_feridos,
    MIN(veiculos) AS min_veiculos,
    MAX(veiculos) AS max_veiculos
FROM workspace.silver.silver_acidentes_prf_2025;

min_pessoas,max_pessoas,min_mortos,max_mortos,min_feridos_leves,max_feridos_leves,min_feridos_graves,max_feridos_graves,min_ilesos,max_ilesos,min_ignorados,max_ignorados,min_feridos,max_feridos,min_veiculos,max_veiculos
1,76,0,16,0,41,0,22,0,71,0,81,0,49,1,82


**Análise de valores extremos:** foram avaliados os valores mínimos e máximos dos atributos quantitativos. Os resultados apresentaram valores entre 1 e 76 pessoas por ocorrência, 0 e 16 mortos, 0 e 41 feridos leves, 0 e 22 feridos graves, 0 e 71 ilesos, 0 e 81 registros ignorados, 0 e 49 feridos e entre 1 e 82 veículos envolvidos. Embora alguns valores máximos sejam elevados em relação à maioria das ocorrências, não foram identificados valores impossíveis com base nas regras de domínio avaliadas. Dessa forma, esses registros foram preservados para as análises.

### 7.2 Solução do Problema

#### 1. Quais são as principais causas dos acidentes registrados nas rodovias federais em 2025?

In [0]:
%sql
SELECT
    a.causa_acidente,
    COUNT(*) AS total_acidentes
FROM workspace.gold.fato_acidentes f
INNER JOIN workspace.gold.dim_acidente a
    ON f.acidente_key = a.acidente_key
GROUP BY a.causa_acidente
ORDER BY total_acidentes DESC
LIMIT 10;

causa_acidente,total_acidentes
Ausência de reação do condutor,11469
Reação tardia ou ineficiente do condutor,10799
Acessar a via sem observar a presença dos outros veículos,7097
Condutor deixou de manter distância do veículo da frente,4413
Velocidade Incompatível,4088
Manobra de mudança de faixa,4016
Ingestão de álcool pelo condutor,3685
Demais falhas mecânicas ou elétricas,3385
Transitar na contramão,2475
Condutor Dormindo,2116


Databricks visualization. Run in Databricks to view.

**Resultado:** A ausência de reação do condutor foi a principal causa de acidentes registrada em 2025, com 11.469 ocorrências, seguida pela reação tardia ou ineficiente do condutor, com 10.799 ocorrências.

#### 2. Em quais dias da semana e horários ocorre o maior número de acidentes?

In [0]:
%sql
SELECT
    t.dia_semana,
    COUNT(*) AS total_acidentes
FROM workspace.gold.fato_acidentes f
INNER JOIN workspace.gold.dim_tempo t
    ON f.tempo_key = t.tempo_key
GROUP BY t.dia_semana
ORDER BY total_acidentes DESC;

dia_semana,total_acidentes
sábado,11554
domingo,11470
sexta-feira,11197
segunda-feira,10285
quarta-feira,9556
quinta-feira,9405
terça-feira,9062


In [0]:
%sql
SELECT
    SUBSTRING(horario, 1, 2) AS hora,
    COUNT(*) AS total_acidentes
FROM workspace.gold.fato_acidentes
GROUP BY SUBSTRING(horario, 1, 2)
ORDER BY total_acidentes DESC;

hora,total_acidentes
18,5398
17,4845
19,4646
07,4583
16,4089
08,3765
15,3690
14,3434
20,3429
06,3169


Databricks visualization. Run in Databricks to view.

**Resultado:** O sábado apresentou o maior número de acidentes, com 11.554 ocorrências. Em relação ao horário, a maior concentração foi registrada às 18h, com 5.398 acidentes, seguida pelas 17h, com 4.845, e 19h, com 4.646 ocorrências.

#### 3. Quais estados e rodovias concentram o maior número de acidentes?

In [0]:
%sql
SELECT
    l.uf,
    COUNT(*) AS total_acidentes
FROM workspace.gold.fato_acidentes f
INNER JOIN workspace.gold.dim_localidade l
    ON f.localidade_key = l.localidade_key
GROUP BY l.uf
ORDER BY total_acidentes DESC
LIMIT 10;

uf,total_acidentes
MG,9570
SC,8186
PR,7630
RJ,6428
RS,4899
SP,4683
BA,4108
GO,3196
PE,3013
ES,2642


In [0]:
%sql
SELECT
    v.br,
    COUNT(*) AS total_acidentes
FROM workspace.gold.fato_acidentes f
INNER JOIN workspace.gold.dim_via v
    ON f.via_key = v.via_key
WHERE v.br <> 0
GROUP BY v.br
ORDER BY total_acidentes DESC
LIMIT 10;

br,total_acidentes
101,13014
116,11021
40,3502
381,3496
153,2789
163,2519
364,2264
277,2157
262,1769
376,1762


**Resultado:** Minas Gerais foi o estado com maior número de acidentes em 2025, com 9.570 ocorrências, seguido por Santa Catarina, com 8.186, e Paraná, com 7.630. Entre as rodovias, a BR-101 apresentou a maior concentração de acidentes, com 13.014 registros, seguida pela BR-116, com 11.021.

#### 4. Quais tipos de acidentes apresentam maior gravidade em relação ao número de vítimas fatais?

In [0]:
%sql
SELECT
    a.tipo_acidente,
    COUNT(*) AS total_acidentes,
    SUM(f.mortos) AS total_mortos
FROM workspace.gold.fato_acidentes f
INNER JOIN workspace.gold.dim_acidente a
    ON f.acidente_key = a.acidente_key
GROUP BY a.tipo_acidente
ORDER BY total_mortos DESC
LIMIT 10;

tipo_acidente,total_acidentes,total_mortos
Colisão frontal,4739,1863
Atropelamento de Pedestre,3057,919
Saída de leito carroçável,10209,700
Colisão traseira,14360,683
Colisão transversal,9306,481
Colisão com objeto,5109,323
Tombamento,6351,293
Colisão lateral sentido oposto,2152,255
Colisão lateral mesmo sentido,7885,228
Queda de ocupante de veículo,3450,89


Databricks visualization. Run in Databricks to view.

**Resultado:** A colisão frontal apresentou o maior número de vítimas fatais, com 1.863 mortes em 4.739 acidentes, seguida pelo atropelamento de pedestre, com 919 mortes em 3.057 ocorrências. Os resultados demonstram que os tipos de acidentes mais frequentes não são necessariamente os que apresentam maior gravidade.

#### 5. Existe relação entre as condições meteorológicas, características da pista e a ocorrência de acidentes?

In [0]:
%sql
SELECT
    a.condicao_metereologica,
    COUNT(*) AS total_acidentes
FROM workspace.gold.fato_acidentes f
INNER JOIN workspace.gold.dim_acidente a
    ON f.acidente_key = a.acidente_key
GROUP BY a.condicao_metereologica
ORDER BY total_acidentes DESC;

condicao_metereologica,total_acidentes
Céu Claro,46375
Nublado,11435
Chuva,6438
Sol,4201
Garoa/Chuvisco,2422
Ignorado,1000
Nevoeiro/Neblina,553
Vento,104
Neve,1


In [0]:
%sql
SELECT
    v.tipo_pista,
    COUNT(*) AS total_acidentes
FROM workspace.gold.fato_acidentes f
INNER JOIN workspace.gold.dim_via v
    ON f.via_key = v.via_key
GROUP BY v.tipo_pista
ORDER BY total_acidentes DESC;

tipo_pista,total_acidentes
Simples,34733
Dupla,30782
Múltipla,7014


**Resultado:** A maior parte dos acidentes ocorreu em condições de céu claro, com 46.375 registros, e em pistas simples, com 34.733 ocorrências. Os dados permitem identificar uma maior concentração de acidentes nessas condições, porém não são suficientes para estabelecer uma relação causal entre condições meteorológicas, tipo de pista e ocorrência de acidentes.


#### 6. Quais características estão mais presentes nos acidentes com vítimas fatais?

In [0]:
%sql
SELECT
    a.tipo_acidente,
    COUNT(*) AS total_acidentes_fatais,
    SUM(f.mortos) AS total_mortos
FROM workspace.gold.fato_acidentes f
INNER JOIN workspace.gold.dim_acidente a
    ON f.acidente_key = a.acidente_key
WHERE a.classificacao_acidente = 'Com Vítimas Fatais'
GROUP BY a.tipo_acidente
ORDER BY total_acidentes_fatais DESC
LIMIT 10;

tipo_acidente,total_acidentes_fatais,total_mortos
Colisão frontal,1395,1862
Atropelamento de Pedestre,902,919
Colisão traseira,619,683
Saída de leito carroçável,605,700
Colisão transversal,427,481
Colisão com objeto,297,323
Tombamento,268,293
Colisão lateral mesmo sentido,212,228
Colisão lateral sentido oposto,212,255
Queda de ocupante de veículo,87,89


**Resultado:** A colisão frontal foi o tipo mais frequente entre os acidentes com vítimas fatais, com 1.395 ocorrências, seguida pelo atropelamento de pedestre, com 902. Esses tipos também apresentaram os maiores números de mortes.

## 8. Conclusão

O projeto implementou um pipeline de dados para análise dos acidentes registrados nas rodovias federais brasileiras em 2025.

Os dados foram ingeridos na Camada Bronze, tratados e padronizados na Camada Silver e organizados na Camada Gold por meio de um modelo dimensional. A partir da estrutura criada, foram realizadas consultas para responder às perguntas de negócio definidas no projeto.

A análise permitiu identificar padrões relacionados às causas, períodos, localidades, rodovias, tipos e gravidade dos acidentes, demonstrando a aplicação das etapas de ingestão, transformação, modelagem e análise de dados em um pipeline no Databricks.

## 9. Autoavaliação

O desenvolvimento deste MVP foi importante para colocar em prática os conceitos estudados na disciplina de Engenharia de Dados. Durante o trabalho, consegui compreender melhor a construção de um pipeline de dados, desde a ingestão até a organização das informações para análise.

Tive algumas dificuldades durante o desenvolvimento, principalmente na preparação e tratamento dos dados e na construção da modelagem dimensional. Também foi necessário realizar ajustes na codificação do arquivo para que os dados fossem apresentados corretamente.

Ao longo do processo, consegui solucionar essas dificuldades e compreender melhor a utilização das camadas Bronze, Silver e Gold e do modelo dimensional em esquema estrela. Considero que o desenvolvimento do MVP contribuiu para o meu aprendizado e para uma melhor compreensão prática dos conceitos apresentados na disciplina.

Considero que os objetivos definidos para o MVP foram atingidos, pois foi possível construir o pipeline de dados proposto e responder às perguntas de negócio estabelecidas no início do projeto.

Como trabalho futuro, o pipeline poderá ser automatizado para incorporar dados de novos períodos, permitindo análises históricas e comparações entre diferentes anos. Também poderão ser incorporadas novas fontes de dados para ampliar as possibilidades de análise dos fatores relacionados aos acidentes.